In [1]:
import pandas as pd
import os
from glob import glob
import warnings
import re

In [ ]:
# 关闭 openpyxl 的无关提醒
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# ====================== 基本配置 ======================
folder_path = "raw_data/Raw_data_value"  # 替换为你的本地路径

# 先拿到所有 xlsx 文件
all_files = glob(os.path.join(folder_path, "*.xlsx"))

# 排除 Excel 的临时锁文件（以 ~$ 开头的）
file_list = [
    fp for fp in all_files
    if not os.path.basename(fp).startswith("~$")
]

print(f"📂 检测到有效文件数量: {len(file_list)}")
for f in file_list:
    print("  -", os.path.basename(f))


# ====================== 核心函数 ======================
def extract_cleaned_data_from_file(file_path: str) -> pd.DataFrame | None:
    """从单个 Excel 文件中提取并清洗数据，返回纵向 DataFrame。"""
    try:
        # 显式指定 engine，避免环境里自动识别出问题
        xls = pd.ExcelFile(file_path, engine="openpyxl")

        # 只保留以 Sheet 开头的 sheet
        sheet_names = [s for s in xls.sheet_names if s.startswith("Sheet")]
        if not sheet_names:
            print(f"⚠️ 跳过：无 Sheet 页 -> {file_path}")
            return None

        frames = []

        for sheet in sheet_names:
            try:
                # 先读原始表头区，用于取元信息
                df = pd.read_excel(xls, sheet_name=sheet, header=None, engine="openpyxl")

                # 提取元信息（多一点容错）
                frequency  = df.iloc[4, 2] if pd.notna(df.iloc[4, 2]) else df.iloc[4, 0]
                product    = df.iloc[5, 2] if pd.notna(df.iloc[5, 2]) else df.iloc[5, 0]
                flow       = df.iloc[6, 2] if pd.notna(df.iloc[6, 2]) else df.iloc[6, 0]
                indicators = df.iloc[7, 2] if pd.notna(df.iloc[7, 2]) else df.iloc[7, 0]

                # 从第 10 行开始读正式数据（header=9，对应 Excel 第 10 行）
                df_data = pd.read_excel(xls, sheet_name=sheet, header=9, engine="openpyxl")

                # 结构校验
                if df_data.shape[0] < 2 or df_data.shape[1] < 2:
                    print(f"⚠️ 跳过 sheet（结构异常）: {sheet} in {file_path}")
                    continue

                # 删除无效第二行（常为冒号）
                df_data = df_data.drop(index=0).reset_index(drop=True)

                # 标准化前两列列名
                df_data.columns.values[0:2] = ["country", "date"]

                # 添加元信息
                df_data["Sheet"] = sheet
                df_data["SourceFile"] = os.path.basename(file_path)
                df_data["Frequency"] = frequency
                df_data["PRODUCT"] = product
                df_data["FLOW"] = flow
                df_data["INDICATORS"] = indicators

                frames.append(df_data)

            except Exception as e:
                print(f"❌ Sheet 跳过: {sheet} in {file_path}\n{e}")
                continue

        if frames:
            return pd.concat(frames, ignore_index=True)
        else:
            print(f"⚠️ 文件跳过（无有效 sheet 数据）: {file_path}")
            return None

    except Exception as e:
        print(f"❌ 文件读取失败: {file_path}\n{e}")
        return None


# ====================== 批量处理所有文件 ======================
df_list = [extract_cleaned_data_from_file(fp) for fp in file_list]
df_list = [df for df in df_list if df is not None]

if df_list:
    # 合并所有文件的数据
    all_data = pd.concat(df_list, ignore_index=True)

    # 宽转长：国家 + 日期 + 元信息 保持不变，其余列转成 partner / value
    metadata_cols = [
        "country", "date",
        "Sheet", "SourceFile",
        "Frequency", "PRODUCT", "FLOW", "INDICATORS"
    ]
    value_cols = [col for col in all_data.columns if col not in metadata_cols]

    df_long = all_data.melt(
        id_vars=metadata_cols,
        value_vars=value_cols,
        var_name="partner",
        value_name="value"
    )

    # 替换冒号为空值，并转为 float
    df_long["value"] = pd.to_numeric(
        df_long["value"].replace({":": None}),
        errors="coerce"
    )

    # 可选：保存结果
    # df_long.to_csv("combined_cleaned_long.csv", index=False)
    # df_long.to_excel("combined_cleaned_long.xlsx", index=False)

    print("✅ 合并后的长表预览：")
    print(df_long.head())
    print(f"✅ 最终行数: {len(df_long)}")

else:
    print("❗ 所有文件均未成功读取或无有效数据。")


📂 检测到有效文件数量: 29
  - data_2022.xlsx
  - data_2018.xlsx
  - data_2014.xlsx
  - data_2025_07_10.xlsx
  - data_2002.xlsx
  - data_2003.xlsx
  - data_2015.xlsx
  - data_2019.xlsx
  - data_2023.xlsx
  - data_2012.xlsx
  - data_2004.xlsx
  - data_2024.xlsx
  - data_2008.xlsx
  - data_2009.xlsx
  - data_2005.xlsx
  - data_2025_11.xlsx
  - data_2013.xlsx
  - data_2006.xlsx
  - data_2025_12.xlsx
  - data_2010.xlsx
  - data_2025_01_06.xlsx
  - data_2011.xlsx
  - data_2007.xlsx
  - data_2020.xlsx
  - data_2000.xlsx
  - data_2016.xlsx
  - data_2017.xlsx
  - data_2001.xlsx
  - data_2021.xlsx
✅ 合并后的长表预览：
   country     date    Sheet      SourceFile Frequency  \
0  Austria  2022-01  Sheet 1  data_2022.xlsx   Monthly   
1  Austria  2022-02  Sheet 1  data_2022.xlsx   Monthly   
2  Austria  2022-03  Sheet 1  data_2022.xlsx   Monthly   
3  Austria  2022-04  Sheet 1  data_2022.xlsx   Monthly   
4  Austria  2022-05  Sheet 1  data_2022.xlsx   Monthly   

                          PRODUCT    FLOW      INDICAT

In [3]:
df_long.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17060610 entries, 0 to 17060609
Data columns (total 10 columns):
 #   Column      Dtype  
---  ------      -----  
 0   country     object 
 1   date        object 
 2   Sheet       object 
 3   SourceFile  object 
 4   Frequency   object 
 5   PRODUCT     object 
 6   FLOW        object 
 7   INDICATORS  object 
 8   partner     object 
 9   value       float64
dtypes: float64(1), object(9)
memory usage: 1.3+ GB


In [4]:
df_long.columns

Index(['country', 'date', 'Sheet', 'SourceFile', 'Frequency', 'PRODUCT',
       'FLOW', 'INDICATORS', 'partner', 'value'],
      dtype='object')

In [5]:
df_long = df_long[['country', 'date', 'PRODUCT', 'INDICATORS', 'partner', 'value']]

In [6]:
def clean_label(value):
    if pd.isna(value):
        return value
    value = re.sub(r"\s*\(.*", "", value)                       # 括号内容
    value = re.sub(r"\s*from\s+\d{4}.*?\)$", "", value)         # from 1994 -> 2000)
    value = re.sub(r"\s*->\s*\d{4}\)?$", "", value)             # -> 1994)
    return value.strip()

In [7]:
# 删除括号及其内容：Belgium (incl. Luxembourg 'LU' -> 1998) → Belgium
df_long["country"] = df_long["country"].apply(clean_label)

df_long["partner"] = df_long["partner"].apply(clean_label)

# date
df_long["date"] = pd.to_datetime(df_long["date"], errors="coerce", format="%Y-%m")
df_long["year"] = df_long["date"].dt.year
df_long["month"] = df_long["date"].dt.month

In [8]:
df_long["date"].unique()

<DatetimeArray>
['2022-01-01 00:00:00', '2022-02-01 00:00:00', '2022-03-01 00:00:00',
 '2022-04-01 00:00:00', '2022-05-01 00:00:00', '2022-06-01 00:00:00',
 '2022-07-01 00:00:00', '2022-08-01 00:00:00', '2022-09-01 00:00:00',
 '2022-10-01 00:00:00',
 ...
 '2021-03-01 00:00:00', '2021-04-01 00:00:00', '2021-05-01 00:00:00',
 '2021-06-01 00:00:00', '2021-07-01 00:00:00', '2021-08-01 00:00:00',
 '2021-09-01 00:00:00', '2021-10-01 00:00:00', '2021-11-01 00:00:00',
 '2021-12-01 00:00:00']
Length: 313, dtype: datetime64[ns]

In [9]:
df_long["PRODUCT"].unique()

array(['New pneumatic tyres, of rubber',
       'New pneumatic tyres, of rubber, of a kind used for motor cars, incl. station wagons and racing cars',
       'New pneumatic tyres, of rubber, of a kind used for buses and lorries (excl. tyres with lug, corner or similar treads)',
       'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of <= 121',
       'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of > 121',
       'New pneumatic tyres, of rubber, of a kind used on agricultural or forestry vehicles and machines',
       'New pneumatic tyres, of rubber, of a kind used on construction, mining or industrial handling vehicles and machines'],
      dtype=object)

In [10]:
hs_code_map = {
    "New pneumatic tyres, of rubber": "4011",
    "New pneumatic tyres, of rubber, of a kind used for motor cars, incl. station wagons and racing cars": "401110",
    "New pneumatic tyres, of rubber, of a kind used for buses and lorries (excl. tyres with lug, corner or similar treads)": "401120",
    'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of <= 121': '40112010',
    'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of > 121': '40112090',
    "New pneumatic tyres, of rubber, of a kind used on agricultural or forestry vehicles and machines": "401170",
    "New pneumatic tyres, of rubber, of a kind used on construction, mining or industrial handling vehicles and machines": "401180",
}

product_map = {
    "New pneumatic tyres, of rubber": "All",
    "New pneumatic tyres, of rubber, of a kind used for motor cars, incl. station wagons and racing cars": "PCR",
    "New pneumatic tyres, of rubber, of a kind used for buses and lorries (excl. tyres with lug, corner or similar treads)": "LT&TBR",
    'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of <= 121': 'LT',
    'Pneumatic tyres, new, of rubber, of a kind used for buses or lorries, with a load index of > 121': 'TBR',
    "New pneumatic tyres, of rubber, of a kind used on agricultural or forestry vehicles and machines": "ARG & FOR",
    "New pneumatic tyres, of rubber, of a kind used on construction, mining or industrial handling vehicles and machines": "GC & Mining & MH",
}

product_map_categoty = {
    "PCR": "PCR",
    "LT": 'PCR',
    "TBR": "TBR",
    "LT&TBR": "LT&TBR",
    "ARG & FOR": "Specialty",
    "GC & Mining & MH": "Specialty",
    'All': 'All'
}

In [11]:
df_long["HS Code"] = df_long["PRODUCT"].map(hs_code_map)
df_long["Category"] = df_long["PRODUCT"].map(product_map)
df_long["General_Category"] = df_long["Category"].map(product_map_categoty)

In [12]:
df_long["HS Code"].unique()

array(['4011', '401110', '401120', '40112010', '40112090', '401170',
       '401180'], dtype=object)

df_long.dropna(subset=['country'], inplace=True)

In [14]:
df_long_filter = df_long[(df_long['country'] != 'Special value')&(df_long['country'] != ':')&(df_long['country'] != 'European Union - 27 countries')]

In [15]:
df_long_filter['partner'].unique()

array(['Andorra', 'United Arab Emirates', 'Afghanistan',
       'Antigua and Barbuda', 'Anguilla', 'Albania', 'Armenia',
       'Netherlands Antilles', 'Angola', 'Antarctica', 'Argentina',
       'American Samoa', 'Austria', 'Australia', 'Aruba', 'Azerbaijan',
       'Bosnia and Herzegovina', 'Barbados', 'Bangladesh', 'Belgium',
       'Burkina Faso', 'Bulgaria', 'Bahrain', 'Burundi', 'Benin',
       'Saint Barthélemy', 'Bermuda', 'Brunei Darussalam',
       'Bolivia, Plurinational State of',
       'Bonaire, Sint Eustatius and Saba', 'Brazil', 'Bahamas', 'Bhutan',
       'Bouvet Island', 'Botswana', 'Belarus', 'Belize', 'Canada',
       'Cocos Islands', 'Congo, Democratic Republic of',
       'Central African Republic', 'Congo', 'Switzerland',
       'Côte d’Ivoire', 'Cook Islands', 'Chile', 'Cameroon', 'China',
       'Colombia', 'Costa Rica', 'Serbia and Montenegro', 'Cuba',
       'Cabo Verde', 'Curaçao', 'Christmas Island', 'Cyprus', 'Czechia',
       'German Democratic Republic',

In [16]:
exclude_values = [
    'All countries of the world',
    'Extra-EU27', 
    'Intra-EU27',
    'Stores and provisions',
    'Stores and provisions within the framework of intra-Union trade',
    'Stores and provisions within the framework of extra-Union trade',
    'Countries and territories not specified',
    'Countries and territories not specified within the framework of intra-Union trade',
    'Countries and territories not specified within the framework of extra-Union trade',
    'Countries and territories not specified for commercial or military reasons',
    'Countries and territories not specified for commercial or military reasons in the framework of intra-Union trade',
    'Countries and territories not specified for commercial or military reasons in the framework of extra-Union trade'
]

df_long_filter = df_long_filter[~df_long_filter['partner'].isin(exclude_values)]

In [17]:
df_long_filter.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16075010 entries, 0 to 17060607
Data columns (total 11 columns):
 #   Column            Dtype         
---  ------            -----         
 0   country           object        
 1   date              datetime64[ns]
 2   PRODUCT           object        
 3   INDICATORS        object        
 4   partner           object        
 5   value             float64       
 6   year              float64       
 7   month             float64       
 8   HS Code           object        
 9   Category          object        
 10  General_Category  object        
dtypes: datetime64[ns](1), float64(3), object(7)
memory usage: 1.4+ GB


In [18]:
# 仅删除 value 列中为 NaN 的行
df_long_filter_drop = df_long_filter.dropna(subset=['value'])

In [19]:
df_long_filter_drop

,country,date,PRODUCT,INDICATORS,partner,value,year,month,HS Code,Category,General_Category
96,Spain,2022-01-01,"New pneumatic tyres, of rubber",VALUE_IN_EUROS,Andorra,2654.0,2022.0,1.0,4011,All,All
97,Spain,2022-02-01,"New pneumatic tyres, of rubber",VALUE_IN_EUROS,Andorra,364.0,2022.0,2.0,4011,All,All
98,Spain,2022-03-01,"New pneumatic tyres, of rubber",VALUE_IN_EUROS,Andorra,3602.0,2022.0,3.0,4011,All,All
99,Spain,2022-04-01,"New pneumatic tyres, of rubber",VALUE_IN_EUROS,Andorra,44.0,2022.0,4.0,4011,All,All
100,Spain,2022-05-01,"New pneumatic tyres, of rubber",VALUE_IN_EUROS,Andorra,1363.0,2022.0,5.0,4011,All,All
...,...,...,...,...,...,...,...,...,...,...,...
17049138,United Kingdom,2000-07-01,"New pneumatic tyres, of rubber, of a kind used...",VALUE_IN_EUROS,Zimbabwe,19411.0,2000.0,7.0,401110,PCR,PCR
17055918,United Kingdom,2001-07-01,"New pneumatic tyres, of rubber",VALUE_IN_EUROS,Zimbabwe,25011.0,2001.0,7.0,4011,All,All
17055923,United Kingdom,2001-12-01,"New pneumatic tyres, of rubber",VALUE_IN_EUROS,Zimbabwe,27822.0,2001.0,12.0,4011,All,All
17056257,United Kingdom,2001-07-01,"New pneumatic tyres, of rubber, of a kind used...",VALUE_IN_EUROS,Zimbabwe,25011.0,2001.0,7.0,401110,PCR,PCR


In [20]:
df_long_filter_drop.to_csv('raw_data/tire_product_Value_PCR_TBR_SP.csv',index=False)